# T5 Nested Biomedical Nested Relation Extractor

## Imports and File Paths

In [1]:
!pip install lark apted

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
from datasets import load_dataset
import json
import random
import os
import wandb
import re
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from lark import Lark, Tree, exceptions
import yaml
from itertools import count
from apted import APTED, Config

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
BASE_PATH = "drive/MyDrive/EPSRC Internship"
DATASET_PATH = os.path.join(BASE_PATH, "nested_relations_dataset_small_v2.json")
SCHEMA_PATH = os.path.join(BASE_PATH, "relation_schema.yml")
TERMS_PATH = os.path.join(BASE_PATH, "terms.json")

ROOT_LABELS = ['effect']

## Model Setup

In [4]:
MAX_LENGTH = 512
MODEL_NAME = "t5-small"
TASK_PREFIX = "extract relations: "
SEED = 42

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

model.resize_token_embeddings(len(tokenizer))  # truncates 32128 -> 32100, drops unused padding rows

print(model.config.vocab_size, len(tokenizer))

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

32100 32100


## Dataset Handling

In [5]:
def mark_entities(sentence, entity_list):
  search_order = sorted(entity_list, key=len, reverse=True)

  first_positions = {}
  for ent in search_order:
    match = re.search(rf'\b{re.escape(ent)}\b', sentence)
    if match:
      first_positions[ent] = match.start()

  ordered_entities = sorted(first_positions, key=first_positions.get)
  entity_to_label = {ent: i+1 for i, ent in enumerate(ordered_entities)}

  marked = sentence
  for ent in search_order:
    if ent not in entity_to_label:
      continue
    label = entity_to_label[ent]
    pattern = rf'\b{re.escape(ent)}\b'
    marked = re.sub(pattern, f'[E{label}] {ent} [/E{label}]', marked)

  return marked


def build_input_text(sentence, entity_list):
    marked = mark_entities(sentence, entity_list)
    entity_suffix = ''.join([f"[SEP]{entity}" for entity in entity_list])
    return TASK_PREFIX + marked + entity_suffix

In [6]:
with open(DATASET_PATH) as f:
    relations = json.load(f)

inputs = []
entities = []
targets = []

for rel in relations:
    input_text = build_input_text(rel["sentence"], rel["entities"])
    inputs.append(input_text)
    entities.append(rel["entities"])
    targets.append(rel["relation_text"])

print(f"Loaded {len(inputs)} examples")

n = len(inputs)
indices = list(range(n))
random.seed(SEED)
random.shuffle(indices)

inputs = [inputs[i] for i in indices]
entities = [entities[i] for i in indices]
targets = [targets[i] for i in indices]

train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_inputs = inputs[:train_split]
val_inputs = inputs[train_split:val_split]
test_inputs = inputs[val_split:]

train_entities = entities[:train_split]
val_entities = entities[train_split:val_split]
test_entities = entities[val_split:]

train_targets = targets[:train_split]
val_targets = targets[train_split:val_split]
test_targets = targets[val_split:]

print(f"Train: {len(train_inputs)}")
print(f"Val:   {len(val_inputs)}")
print(f"Test:  {len(test_inputs)}")

Loaded 3551 examples
Train: 2840
Val:   355
Test:  356


In [7]:
def make_hf_dataset(inputs, targets):
    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

raw_datasets = DatasetDict({
    "train": make_hf_dataset(train_inputs, train_targets),
    "validation": make_hf_dataset(val_inputs, val_targets),
    "test": make_hf_dataset(test_inputs, test_targets),
})


def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=512,
        truncation=True,
    )
    labels = tokenizer(
        examples["target_text"],
        max_length=512,
        truncation=True,
    )
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"],
)

Map:   0%|          | 0/2840 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

Map:   0%|          | 0/356 [00:00<?, ? examples/s]

## Metrics

### Parser and Schema Checker

In [8]:
grammar = r"""
  start: predicate

  predicate: NAME "(" [arg ("," arg)*] ")"

  arg: NAME "=" value

  value: predicate
     | STRING

  NAME: /[a-zA-Z_][a-zA-Z0-9_]*/
  STRING: /(?s)\[QUOTE\].*?\[QUOTE\]/

  %import common.WS
  %ignore WS
"""

parser = Lark(grammar, start="start", parser="earley")

In [9]:
def parse_tree(text):
    try:
        tree = parser.parse(text)
        return tree, None
    except exceptions.UnexpectedCharacters as e:
        return None, f"unexpected_char at pos {e.pos_in_stream}: {e}"
    except exceptions.UnexpectedEOF as e:
        return None, "truncated_input"
    except exceptions.UnexpectedToken as e:
        return None, f"unexpected_token at pos {e.pos_in_stream}: {e}"
    except exceptions.LarkError as e:
        return None, f"parse_error: {e}"

def parse_error_category(err):
    if err is None:
        return "success"
    if err == "truncated_input":
        return "truncated_input"
    if err.startswith("unexpected_char"):
        return "unexpected_char"
    if err.startswith("unexpected_token"):
        return "unexpected_token"
    return "other_parse_error"

In [10]:
def load_terms(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    terms = {category: set(entities) for category, entities in raw.items()}
    terms['biomolecule'] = terms['chemical'].union(terms['gene'])
    terms['gene_protein'] = terms['gene']
    return terms


def load_relation_schema(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def strip_quote_markers(string_token_text):
    return string_token_text[len("[QUOTE]"):-len("[QUOTE]")]

In [11]:
def validate_predicate(predicate_tree, allowed_types, relation_schema, terms):
    label = predicate_tree.children[0].value
    arg_nodes = [c for c in predicate_tree.children[1:] if c is not None]

    if label not in relation_schema:
        return False, f"unknownn_relation_label: '{label}' is not defined in relation_schema"

    if allowed_types is not None and label not in allowed_types:
        return False, (
            f"disallowed_relation_type: '{label}' is not permitted in this position "
            f"(allowed: {allowed_types})"
        )

    definition = relation_schema[label]
    required_args = set(definition["args"])

    present = {}
    for arg_node in arg_nodes:
        arg_name = arg_node.children[0].value
        arg_tree = arg_node.children[1]
        present[arg_name] = arg_tree

    present_names = set(present.keys())

    missing = required_args - present_names
    if missing:
        return False, f"missing_args: '{label}' is missing required arg(s) {sorted(missing)}"

    unexpected = present_names - required_args
    if unexpected:
        return False, f"unexpected_args: '{label}' has unrecognized arg(s) {sorted(unexpected)}"

    for arg_name in required_args:
        arg_tree = present[arg_name]
        arg_types = definition.get("arg_types", {}).get(arg_name, [arg_name])
        ok, err = validate_arg(arg_tree, arg_types, relation_schema, terms)
        if not ok:
            return False, f"in {label}.{arg_name}: {err}"

    return True, None


def validate_arg(arg_tree, allowed_types, relation_schema, terms):
    inner = arg_tree.children[0]

    if isinstance(inner, Tree) and inner.data == "predicate":
        return validate_predicate(inner, allowed_types, relation_schema, terms)

    entity_str = strip_quote_markers(str(inner))
    entity_categories = [t for t in allowed_types if t in terms]

    for cat in entity_categories:
        if entity_str in terms[cat]:  # O(1): terms[cat] is a set
            return True, None

    if not entity_categories:
        return False, (
            f"unexpected_entity_position: got a bare entity '{entity_str}' but none of the "
            f"allowed types here are entity categories (allowed_types={allowed_types})"
        )
    return False, (
        f"unknown_entity: '{entity_str}' not found in any allowed category {entity_categories}"
    )

### Binary Decomposition

In [12]:
def decompose(predicate_tree, binary_rels=None, ids=None, current=None):
    if binary_rels is None:
        binary_rels = []
        ids = count()

    if current is None:
        current = (next(ids), predicate_tree.children[0].value)

    for arg_node in predicate_tree.children[1:]:
        arg_name = arg_node.children[0].value
        inner = arg_node.children[1].children[0]

        if isinstance(inner, Tree) and inner.data == "predicate":
            child = (next(ids), inner.children[0].value)
            binary_rels.append((current, arg_name, child))
            decompose(inner, binary_rels, ids, child)
        else:
            entity = strip_quote_markers(str(inner))
            binary_rels.append((current, arg_name, entity))

    return set(binary_rels)

In [13]:
def bin_decompostion_metrics(pred_trees, target_trees):
    TP = 0
    FP = 0
    FN = 0

    for pred_tree, target_tree in zip(pred_trees, target_trees):
        pred_bin_rels = decompose(pred_tree)
        target_bin_rels = decompose(target_tree)

        TP += len(target_bin_rels & pred_bin_rels)
        FP += len(pred_bin_rels.difference(target_bin_rels))
        FN += len(target_bin_rels.difference(pred_bin_rels))

    precision = TP / (TP+FP) if (TP+FP) else 0
    recall = TP / (TP+FN) if (TP+FN) else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        "bin_decomp_precision" : precision,
        "bin_decomp_recall" : recall,
        "bin_decomp_f1_score" : f1_score,
    }

### Tree Edit Distance

In [14]:
class TedNode:
    __slots__ = ("label", "children")

    def __init__(self, label, children=None):
        self.label = label
        self.children = children or []

    def __str__(self):
          return ted_node_to_str(self)

def ted_node_to_str(node, indent=0):
    pad = "  " * indent
    lines = [f"{pad}{node.label}"]
    for child in node.children:
        lines.append(ted_node_to_str(child, indent + 1))
    return "\n".join(lines)

def to_ted_tree(predicate_tree):
    label = predicate_tree.children[0].value
    node = TedNode(label)

    for arg_node in predicate_tree.children[1:]:
        if arg_node is None:
            continue
        arg_name = arg_node.children[0].value
        inner = arg_node.children[1].children[0]

        role_node = TedNode(f"@{arg_name}")
        if isinstance(inner, Tree) and inner.data == "predicate":
            role_node.children = [to_ted_tree(inner)]
        else:
            entity = strip_quote_markers(str(inner))
            role_node.children = [TedNode(f"={entity}")]

        node.children.append(role_node)

    node.children.sort(key=lambda c: c.label)
    return node

In [15]:
class TedConfig(Config):
    def rename(self, node1, node2):
        return 0 if node1.label == node2.label else 1  # unit-cost relabel
    def children(self, node):
        return node.children
    def insert(self, node):
        return 1  # unit-cost insert
    def delete(self, node):
        return 1  # unit-cost delete

def tree_edit_distance(pred_tree, gold_tree):
    pred_ted = to_ted_tree(pred_tree)
    gold_ted = to_ted_tree(gold_tree)
    return APTED(pred_ted, gold_ted, TedConfig()).compute_edit_distance()

def tree_size(node):
    return 1 + sum(tree_size(c) for c in node.children)

def normalized_ted(pred_tree, gold_tree):
    pred_ted = to_ted_tree(pred_tree)
    gold_ted = to_ted_tree(gold_tree)
    dist = APTED(pred_ted, gold_ted, TedConfig()).compute_edit_distance()
    max_size = max(tree_size(pred_ted), tree_size(gold_ted))
    return 1 - dist / max_size

In [16]:
def mean_norm_tree_edit_distance(pred_trees, target_trees):
    n = len(pred_trees)
    cum_norm_ted = 0

    for pred_tree, target_tree in zip(pred_trees, target_trees):
        cum_norm_ted += normalized_ted(pred_tree, target_tree)

    mean_norm_ted = cum_norm_ted / n if n else 0.0

    return mean_norm_ted

### Metric Calculation

In [ ]:
def parse_and_evaluate(preds, targets, root_labels=ROOT_LABELS):
    n = len(preds)

    parsed_targets = [parse_tree(t) for t in targets]
    parsed_trees = [parse_tree(p) for p in preds]
    parse_rate = sum(1 for _, err in parsed_trees if err is None) / n if n > 0 else 0.0

    valid_parse_preds_targets = [
        (pred_tree.children[0], target_tree.children[0])
        for (pred_tree, perr), (target_tree, terr) in zip(parsed_trees, parsed_targets)
        if perr is None and terr is None
    ]

    relation_schema = load_relation_schema(SCHEMA_PATH)
    terms = load_terms(TERMS_PATH)

    valid_schema_pred_trees = []
    valid_schema_target_trees = []
    n_after_parse = len(valid_parse_preds_targets)
    for (pred_tree, target_tree) in valid_parse_preds_targets:
        ok, err = validate_predicate(pred_tree, root_labels, relation_schema, terms)
        if ok:
            valid_schema_pred_trees.append(pred_tree)
            valid_schema_target_trees.append(target_tree)

    schema_rate = len(valid_schema_pred_trees) / n_after_parse if n_after_parse > 0 else 0.0

    bin_decomp_results = bin_decompostion_metrics(valid_schema_pred_trees, valid_schema_target_trees)

    mean_norm_ted = mean_norm_tree_edit_distance(valid_schema_pred_trees, valid_schema_target_trees)

    return {
        "parse_rate": parse_rate,
        "schema_rate": schema_rate,
        **bin_decomp_results,
        "mean_norm_ted": mean_norm_ted

    }

## Training and Evaluation

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Replace -100 in preds and labels (padding token used by data collator) before decoding
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    return parse_and_evaluate(decoded_preds, decoded_labels)

In [23]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-relation-extraction",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    predict_with_generate=True,
    generation_max_length=512,
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    metric_for_best_model="bin_decomp_f1_score",
    greater_is_better=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [ ]:
trainer.evaluate()